# sqljudge overnight runs on Colab (setup C)
Same harness and settings as the PC runs: Ollama 0.40.4, context 16,384, 20 tool calls, all 1,000 ClaimDB public-test claims.

**One time:** upload `sqljudge_colab_bundle.zip` to Google Drive at `MyDrive/sqljudge_colab/`.

**Every session:** Runtime → Change runtime type → Python 3, **L4 GPU**. Then Runtime → Run all, approve the Google Drive popup, and leave the tab open.

Every claim is saved to Drive the moment it finishes. If Colab disconnects, open the notebook again and Run all: each run continues where it stopped, finished runs are skipped.

In [ ]:
# 1. Job list, in order (Kanishk's part of the execution plan)
JOBS = [
    # finish Section 1
    {"judge": "qwen3:14b",   "builder": "hf.co/mradermacher/Arctic-Text2SQL-R1-7B-GGUF:Q8_0"},
    # Section 3 grid cells
    {"judge": "qwen3:4b",    "builder": "hf.co/mradermacher/OmniSQL-7B-GGUF:Q8_0"},
    {"judge": "llama3.1:8b", "builder": "hf.co/mradermacher/OmniSQL-7B-GGUF:Q8_0"},
    {"judge": "qwen3:4b",    "builder": "qwen2.5-coder:7b-instruct-q8_0"},
    {"judge": "qwen3:4b",    "builder": "cycloneboy/SLM-SQL-1.5B", "backend": "hf"},
    {"judge": "llama3.1:8b", "builder": "cycloneboy/SLM-SQL-1.5B", "backend": "hf"},
]

In [ ]:
# 2. GPU check
import subprocess
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print(gpu or 'NO GPU: Runtime > Change runtime type > L4 GPU')
assert gpu, 'No GPU attached'

In [ ]:
# 3. Google Drive (approve the popup once; it stays connected for the whole session)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, glob, json
BASE = '/content/drive/MyDrive/sqljudge_colab'
os.makedirs(f'{BASE}/results', exist_ok=True); os.makedirs(f'{BASE}/logs', exist_ok=True)
assert os.path.exists(f'{BASE}/sqljudge_colab_bundle.zip'), 'Upload sqljudge_colab_bundle.zip to MyDrive/sqljudge_colab first'

In [ ]:
# 4. Code + earlier partial results (never overwrites results already in Drive)
!rm -rf /content/sqljudge && mkdir -p /content/sqljudge && unzip -q -o "$BASE/sqljudge_colab_bundle.zip" -d /content/sqljudge
for f in glob.glob('/content/sqljudge/seed_results/*.jsonl'):
    dst = f"{BASE}/results/{os.path.basename(f)}"
    if not os.path.exists(dst):
        shutil.copy(f, dst); print('seeded', os.path.basename(f))
!pip -q install -r /content/sqljudge/code/requirements.txt

In [ ]:
# 5. BIRD dev databases (official BIRD download, cached in Drive after the first time)
if not os.path.exists(f'{BASE}/bird_dev.zip'):
    !wget -q -O /content/bird_dev.zip https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip && cp /content/bird_dev.zip "$BASE/bird_dev.zip"
!mkdir -p /content/bird && unzip -q -o "$BASE/bird_dev.zip" -d /content/bird
for z in glob.glob('/content/bird/**/dev_databases.zip', recursive=True):
    subprocess.run(['unzip','-q','-o',z,'-d','/content/bird/db'], check=True)
need = ['california_schools','card_games','codebase_community','debit_card_specializing','european_football_2',
        'financial','formula_1','student_club','superhero','thrombosis_prediction','toxicology']
found = {os.path.basename(os.path.dirname(p)): os.path.dirname(os.path.dirname(p))
         for p in glob.glob('/content/bird/db/**/*.sqlite', recursive=True)}
missing = [d for d in need if d not in found]; assert not missing, f'missing databases: {missing}'
DB_ROOT = found[need[0]]; print('all 11 databases found at', DB_ROOT)

In [ ]:
# 6. Ollama, same version as the PC (0.40.4), same context length
!apt-get -qq install -y pciutils lshw zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=0.40.4 sh > /content/ollama_install.log 2>&1 || curl -fsSL https://ollama.com/install.sh | sh > /content/ollama_install.log 2>&1
import time
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH='16384', OLLAMA_MAX_LOADED_MODELS='2')
subprocess.Popen('nohup ollama serve > /content/ollama.log 2>&1 &', shell=True, env=env)
time.sleep(10)
!ollama --version

In [ ]:
# 7. Start the job queue in the background
json.dump(JOBS, open('/content/jobs.json', 'w'))
subprocess.Popen(f'nohup python -u /content/sqljudge/code/colab_queue.py --jobs /content/jobs.json --drive "{BASE}" '
                 f'--db-root "{DB_ROOT}" > /content/queue.out 2>&1 &', shell=True, env=env)
time.sleep(5)
!tail -n 5 "$BASE/queue_log.txt"

In [ ]:
# 8. Monitor: keeps running (and keeps the session active) until every job is done. Leave it running.
import datetime
def safe(s): return s.replace(':','_').replace('/','_')
def report():
    lines = []
    for j in JOBS:
        p = f"{BASE}/results/{safe(j['judge'])}__C__{safe(j['builder'])}.jsonl"
        R = []
        if os.path.exists(p):
            for l in open(p, encoding='utf-8'):
                try: R.append(json.loads(l))
                except Exception: pass
        acc = f"{100*sum(r['verdict']==r['label'] for r in R)/len(R):5.1f}%" if R else '   -  '
        lines.append(f"{j['judge']:12s} + {j['builder'].split('/')[-1][:32]:32s} {len(R):4d}/1000  acc {acc}")
    return lines
while True:
    print(datetime.datetime.now().strftime('%H:%M'), '-'*40); print('\n'.join(report()))
    log = open(f'{BASE}/queue_log.txt').read().splitlines() if os.path.exists(f'{BASE}/queue_log.txt') else []
    print('last log:', log[-1] if log else '-', flush=True)
    if log and log[-1].endswith('queue finished'): break
    time.sleep(600)

### After a disconnect
Open the notebook and Runtime → Run all again. Finished jobs are skipped and the running one continues from its last saved claim.